# GATE: prior de ARC de Qwen3.5-35B-A3B (stock) vs Qwen3-4B (stock)

Antes de comprometer 2-4 semanas de presupuesto en SFT+TTT sobre el MoE (TTT_PLAN 49 ter), la
pregunta barata: **el conocimiento de 35B ayuda en rejillas, o solo disparan pocos expertos y la
capacidad efectiva es ~3B?**

Medida: **NLL por celda de la rejilla correcta**, con las demostraciones en el prompt como texto,
para ambos modelos **de stock** (sin SFT de ARC, sin TTT, sin cirugia de vocabulario). Comparacion
pareada sobre las mismas 24 tareas del eval limpias de AGI-1. Se normaliza por celda porque los
tokenizadores pueden diferir. Se mide ademas el tiempo de forward de cada modelo sobre el mismo
prompt: valida (o refuta) la aritmetica de bandwidth de 49 bis.

El MoE se carga en **4-bit NF4** (17,5 GB) en una L4; el 4B en bf16. Cada worker corre ambos
secuencialmente sobre su shard. El MoE es un VLM: se prueban varias clases de carga con fallback,
y si no carga, el 4B se mide igual y el fallo queda registrado.

**PRERREG**: ratio mediano nll/celda MoE/4B <= 0,80 **y** el MoE gana en >= 2/3 de las tareas =>
mejor prior, el proyecto sigue. Ratio ~1 o gana < 2/3 => el MoE no compra nada => proyecto muerto.


In [ ]:
%%writefile /kaggle/working/_worker_src.py
import gc, json, os, sys, time, math
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T00 = time.time()
SHARD = int(os.environ.get("SHARD", "0")); N_SHARDS = int(os.environ.get("N_SHARDS", "4"))
N_TAREAS = int(os.environ.get("N_TAREAS", "24"))
SUCIAS = set(json.loads(os.environ["SUCIAS"]))
OUT = "/kaggle/working/shard%d.json" % SHARD

import torch

FICH = "arc-agi_evaluation_challenges.json"
DATA = None
for cand in ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2", "/kaggle/input/arc-prize-2026-arc-agi-2"):
    if os.path.exists(os.path.join(cand, FICH)):
        DATA = cand
        break
if DATA is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if FICH in files:
            DATA = root
            break
assert DATA, "sin dataset; /kaggle/input: %s" % os.listdir("/kaggle/input")
CHAL = json.load(open(os.path.join(DATA, FICH)))
SOLS = json.load(open(os.path.join(DATA, "arc-agi_evaluation_solutions.json")))

def buscar_modelo(*marcas):
    for root, dirs, files in os.walk("/kaggle/input/models"):
        low = root.lower().replace("\\", "/")
        if "config.json" in files and all(m in low for m in marcas):
            return root
    return None
DIR_4B  = buscar_modelo("qwen-3/", "/4b")
DIR_MOE = buscar_modelo("qwen3.5-35b-a3b")
print("[s%d] 4B en %s | MoE en %s" % (SHARD, DIR_4B, DIR_MOE), flush=True)
assert DIR_4B, "no se encontro Qwen3-4B de stock"

def coste(t):
    return sum(len(g["input"])*len(g["input"][0]) + len(g["output"])*len(g["output"][0]) for g in t["train"])
KEYS_ALL = sorted([k for k in CHAL if k not in SUCIAS], key=lambda k: coste(CHAL[k]))[:N_TAREAS]
MIS = KEYS_ALL[SHARD::N_SHARDS]
print("[s%d] %d tareas" % (SHARD, len(MIS)), flush=True)

def rejilla_txt(g):
    return "\n".join(" ".join(str(c) for c in fila) for fila in g)

def prompt_y_gt(task, gt):
    p = "Solve the puzzle. Each example maps an input grid to an output grid by the same rule.\n\n"
    for i, par in enumerate(task["train"]):
        p += "Example %d\nInput:\n%s\nOutput:\n%s\n\n" % (i+1, rejilla_txt(par["input"]), rejilla_txt(par["output"]))
    p += "Test\nInput:\n%s\nOutput:\n" % rejilla_txt(task["test"][0]["input"])
    return p, rejilla_txt(gt) + "\n"

@torch.no_grad()
def nll_gt(model, tok, prompt, gt_txt):
    ids_p = tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids
    ids_g = tok(gt_txt, return_tensors="pt", add_special_tokens=False).input_ids
    ids = torch.cat([ids_p, ids_g], dim=1).to(next(model.parameters()).device)
    if ids.shape[1] > 12000:
        return None
    t0 = time.time()
    out = model(input_ids=ids, use_cache=False, return_dict=True)
    torch.cuda.synchronize()
    dt = time.time() - t0
    lg = out.logits[0, :-1].float()
    tgt = ids[0, 1:]
    lp = lg.log_softmax(-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    n_p = ids_p.shape[1]
    nll = float(-(lp[n_p-1:]).sum().item())      # solo los tokens de la GT
    return {"nll": nll, "n_tok_gt": int(ids_g.shape[1]), "n_tok_total": int(ids.shape[1]), "fwd_s": dt}

def cargar(dir_, cuatro_bits):
    import transformers
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(dir_, trust_remote_code=True)
    kw = dict(trust_remote_code=True, low_cpu_mem_usage=True)
    modo = "bf16-1gpu"
    if cuatro_bits:
        try:
            import bitsandbytes  # noqa: F401
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.bfloat16,
                                                           bnb_4bit_use_double_quant=True)
            kw["device_map"] = {"": 0}
            modo = "nf4-1gpu"
        except Exception as e:
            # sin bitsandbytes: bf16 repartido en las 4 GPUs (70 GB en 96). Vale para medir el
            # PRIOR; no vale para medir la velocidad del 4-bit (se anota).
            print("[s%d] sin bitsandbytes (%s) -> bf16 device_map=auto" % (SHARD, str(e)[:80]), flush=True)
            kw["torch_dtype"] = torch.bfloat16
            kw["device_map"] = "auto"
            modo = "bf16-auto"
    else:
        kw["torch_dtype"] = torch.bfloat16
        kw["device_map"] = {"": 0}
    print("[s%d] modo de carga: %s" % (SHARD, modo), flush=True)
    errores = []
    for nombre in ("AutoModelForCausalLM", "AutoModelForImageTextToText", "AutoModelForVision2Seq", "AutoModel"):
        try:
            cls = getattr(transformers, nombre)
            m = cls.from_pretrained(dir_, **kw)
            m.eval()
            print("[s%d] cargado con %s" % (SHARD, nombre), flush=True)
            return m, tok, nombre
        except Exception as e:
            errores.append("%s: %s" % (nombre, str(e)[:160]))
    raise RuntimeError(" | ".join(errores))

RES = {"filas": [], "shard": SHARD, "errores": {}}
tareas = {k: (CHAL[k], SOLS[k][0]) for k in MIS}

for etiq, dir_, q4 in (("4b", DIR_4B, False), ("moe", DIR_MOE, True)):
    if dir_ is None:
        RES["errores"][etiq] = "directorio no encontrado"
        continue
    t0 = time.time()
    try:
        model, tok, clsn = cargar(dir_, q4)
    except Exception as e:
        RES["errores"][etiq] = str(e)[:600]
        print("[s%d] %s NO CARGA: %s" % (SHARD, etiq, str(e)[:300]), flush=True)
        json.dump(RES, open(OUT, "w"))
        continue
    mem = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 1e9
    print("[s%d] %s listo: %.1f GB, carga %.0fs (%s)" % (SHARD, etiq, mem, time.time()-t0, clsn), flush=True)
    for k, (task, gt) in tareas.items():
        p, g = prompt_y_gt(task, gt)
        try:
            r = nll_gt(model, tok, p, g)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            r = None
        except Exception as e:
            print("[s%d] %s %s error: %s" % (SHARD, etiq, k, str(e)[:120]), flush=True)
            r = None
        celdas = len(gt) * len(gt[0])
        fila = {"task": k, "modelo": etiq, "celdas": celdas, "mem_gb": round(mem, 2)}
        if r:
            fila.update(r)
            fila["nll_por_celda"] = round(r["nll"] / celdas, 4)
            fila["tok_por_celda"] = round(r["n_tok_gt"] / celdas, 3)
        RES["filas"].append(fila)
        print("[s%d] %-4s %s: nll/celda %s | tok/celda %s | fwd %ss" % (SHARD, etiq, k,
              fila.get("nll_por_celda", "-"), fila.get("tok_por_celda", "-"),
              ("%.2f" % r["fwd_s"]) if r else "-"), flush=True)
        json.dump(RES, open(OUT, "w"))
    del model
    gc.collect()
    torch.cuda.empty_cache()

json.dump(RES, open(OUT, "w"))
print("[s%d] fin %.0fs" % (SHARD, time.time() - T00), flush=True)


In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
SUCIAS = ["0934a4d8", "136b0064", "16b78196", "981571dc", "aa4ec2a5", "da515329"]
open("/kaggle/working/worker.py", "w").write(open("/kaggle/working/_worker_src.py").read())
# v3: UN solo worker con las 4 GPUs visibles. El MoE en bf16 son 70 GB y se reparte con
# device_map=auto; para 24 forwards de medicion no hace falta paralelismo. (Excepcion a la
# regla de shardear x4: aqui la maquina entera la usa UN modelo, no queda GPU ociosa.)
procs = []
for i in range(1):
    env = dict(os.environ, SHARD="0", N_SHARDS="1", N_TAREAS="24",
               SUCIAS=json.dumps(SUCIAS), PYTHONUNBUFFERED="1")
    p = subprocess.Popen([sys.executable, "/kaggle/working/worker.py"], env=env,
                         stdout=open("/kaggle/working/log%d.txt" % i, "w"), stderr=subprocess.STDOUT)
    procs.append(p)
    print("lanzado worker %d (4 GPUs visibles)" % i, flush=True)
for i, p in enumerate(procs):
    p.wait()
    print("worker %d rc=%s (%.0fs)" % (i, p.returncode, time.time()-T0), flush=True)

filas, errores = [], {}
for i in range(1):
    f = "/kaggle/working/shard%d.json" % i
    if os.path.exists(f):
        try:
            d = json.load(open(f))
            filas += d["filas"]
            errores.update({"s%d_%s" % (i, k): v for k, v in d.get("errores", {}).items()})
        except Exception as e:
            print("shard %d ilegible: %s" % (i, e), flush=True)
json.dump({"filas": filas, "errores": errores}, open("/kaggle/working/gate_moe.json", "w"))
for i in range(1):
    print("")
    print("=" * 14 + (" LOG WORKER %d " % i) + "=" * 14, flush=True)
    try:
        print(open("/kaggle/working/log%d.txt" % i).read()[-2000:], flush=True)
    except Exception as e:
        print("sin log: %s" % e, flush=True)

por = {}
for f in filas:
    if "nll_por_celda" in f:
        por.setdefault(f["task"], {})[f["modelo"]] = f
pares = [(t, d["4b"], d["moe"]) for t, d in por.items() if "4b" in d and "moe" in d]
print("")
print("===== GATE: prior de ARC, Qwen3-4B stock vs Qwen3.5-35B-A3B stock =====", flush=True)
print("%-10s %10s %10s %8s | %8s %8s" % ("tarea", "nll/c 4B", "nll/c MoE", "ratio", "fwd 4B", "fwd MoE"), flush=True)
gana = 0
ratios = []
for t, a, b in sorted(pares):
    r = b["nll_por_celda"] / max(a["nll_por_celda"], 1e-9)
    ratios.append(r)
    gana += (r < 1)
    print("%-10s %10.3f %10.3f %8.2f | %7.2fs %7.2fs" % (t, a["nll_por_celda"], b["nll_por_celda"], r, a["fwd_s"], b["fwd_s"]), flush=True)
if pares:
    ratios.sort()
    med = ratios[len(ratios)//2]
    f4 = sum(a["fwd_s"] for _, a, _ in pares) / len(pares)
    fm = sum(b["fwd_s"] for _, _, b in pares) / len(pares)
    print("")
    print("MoE gana en %d/%d tareas | ratio MEDIANO nll/celda MoE/4B = %.2f" % (gana, len(pares), med), flush=True)
    print("forward medio: 4B %.2fs | MoE %.2fs -> MoE es %.2fx" % (f4, fm, fm / max(f4, 1e-9)), flush=True)
    print("PRERREG: ratio mediano <= 0,80 Y gana >= 2/3 => el MoE tiene MEJOR prior => proyecto vivo", flush=True)
    print("         ratio ~1 o gana < 2/3 => no compra nada => proyecto MUERTO por ~1 h-cuota", flush=True)
if errores:
    print("ERRORES:", json.dumps(errores)[:1500], flush=True)
print("RESULT_JSON:" + json.dumps({"n_pares": len(pares), "gana_moe": gana, "errores": len(errores), "wall_s": round(time.time()-T0)}), flush=True)
